In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import get_prices
from src.backtest import run_backtest

In [2]:
tickers = ["AAPL", "TSLA", "SPY"]
prices = get_prices(tickers)

portfolio_returns, weights_df = run_backtest(
    prices,
    window=60,
    rebalance_freq=21,
    transaction_cost=0.0001
)

[*********************100%***********************]  3 of 3 completed
ERROR:root:Inputs must be numpy arrays.
ERROR:root:Inputs must be numpy arrays.


RuntimeError: Error in backtest loop at index 63: '<' not supported between instances of 'NoneType' and 'float'

In [ ]:
# Get regime information from the prices data
from src.regime import detect_regime
regimes = detect_regime(prices)
regime_series = regimes.iloc[60:]  # Match the window size

# Align the regime series with portfolio returns
regime_aligned = regime_series.iloc[1:]  # Adjust for the first return being skipped

results = pd.DataFrame({
    "returns": portfolio_returns,
    "regime": regime_aligned
})

In [ ]:
bull = results[results["regime"] == "bull"]
bear = results[results["regime"] == "bear"]

In [ ]:
def metrics(r):
    return {
        "mean_return": r.mean(),
        "volatility": r.std(),
        "sharpe": r.mean() / r.std()
    }

In [ ]:
print("BULL MARKET METRICS")
print(metrics(bull["returns"]))

print("\nBEAR MARKET METRICS")
print(metrics(bear["returns"]))

In [ ]:
results["cum_return"] = (1 + results["returns"]).cumprod()

In [ ]:
bull_contrib = (1 + bull["returns"]).prod() - 1
bear_contrib = (1 + bear["returns"]).prod() - 1

print("Bull Contribution:", bull_contrib)
print("Bear Contribution:", bear_contrib)

In [ ]:
plt.figure(figsize=(10,5))

plt.plot((1 + results["returns"]).cumprod(), label="Portfolio")

plt.title("Equity Curve with Regime-Aware Strategy")
plt.legend()
plt.show()

In [ ]:
summary = pd.DataFrame({
    "Bull": metrics(bull["returns"]),
    "Bear": metrics(bear["returns"])
})

summary

In [3]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import get_prices
from src.backtest import run_backtest

# Load data
tickers = ["AAPL", "TSLA", "SPY"]
prices = get_prices(tickers)

# Ensure we have sufficient data
print(f"Available data points: {len(prices)}")
print(f"Data columns: {list(prices.columns)}")

# Run backtest with error handling
try:
    portfolio_returns, weights_df = run_backtest(
        prices,
        window=60,
        rebalance_freq=21,
        transaction_cost=0.0001
    )
    
    print(f"Backtest completed successfully")
    print(f"Portfolio returns length: {len(portfolio_returns)}")
    print(f"Weights DataFrame shape: {weights_df.shape}")
    
except Exception as e:
    print(f"Error running backtest: {str(e)}")
    # Try with different parameters
    try:
        portfolio_returns, weights_df = run_backtest(
            prices,
            window=30,
            rebalance_freq=10,
            transaction_cost=0.0001
        )
        print("Backtest completed with reduced parameters")
    except Exception as e2:
        print(f"Error with reduced parameters: {str(e2)}")
        raise

# Get regime information from the prices data
try:
    from src.regime import detect_regime
    regimes = detect_regime(prices)
    regime_series = regimes.iloc[60:]  # Match the window size
    
    # Align the regime series with portfolio returns
    regime_aligned = regime_series.iloc[1:]  # Adjust for the first return being skipped
    
    results = pd.DataFrame({
        "returns": portfolio_returns,
        "regime": regime_aligned
    })
    
    print(f"Results DataFrame shape: {results.shape}")
    print(f"Regime distribution:\n{results['regime'].value_counts()}")
    
except Exception as e:
    print(f"Error in regime analysis: {str(e)}")
    raise

# Perform analysis
try:
    bull = results[results["regime"] == "bull"]
    bear = results[results["regime"] == "bear"]
    
    def metrics(r):
        return {
            "mean_return": r.mean(),
            "volatility": r.std(),
            "sharpe": r.mean() / r.std() if r.std() != 0 else 0
        }
    
    print("BULL MARKET METRICS")
    print(metrics(bull["returns"]))
    
    print("\nBEAR MARKET METRICS")
    print(metrics(bear["returns"]))
    
    results["cum_return"] = (1 + results["returns"]).cumprod()
    
    bull_contrib = (1 + bull["returns"]).prod() - 1
    bear_contrib = (1 + bear["returns"]).prod() - 1
    
    print("Bull Contribution:", bull_contrib)
    print("Bear Contribution:", bear_contrib)
    
    plt.figure(figsize=(10,5))
    plt.plot((1 + results["returns"]).cumprod(), label="Portfolio")
    plt.title("Equity Curve with Regime-Aware Strategy")
    plt.legend()
    plt.show()
    
    summary = pd.DataFrame({
        "Bull": metrics(bull["returns"]),
        "Bear": metrics(bear["returns"])
    })
    
    print("\nSummary DataFrame:")
    print(summary)
    
except Exception as e:
    print(f"Error in analysis: {str(e)}")
    raise

[*********************100%***********************]  3 of 3 completed
ERROR:root:Inputs must be numpy arrays.
ERROR:root:Inputs must be numpy arrays.
ERROR:root:Inputs must be numpy arrays.
ERROR:root:Inputs must be numpy arrays.


Available data points: 1006
Data columns: ['AAPL', 'SPY', 'TSLA']
Error running backtest: Error in backtest loop at index 63: '<' not supported between instances of 'NoneType' and 'float'
Error with reduced parameters: Error in backtest loop at index 30: '<' not supported between instances of 'NoneType' and 'float'


RuntimeError: Error in backtest loop at index 30: '<' not supported between instances of 'NoneType' and 'float'